Assignment 2
Stage 1
Krishnaveni Varkal

This code trains a network to catagorize movie reviews as either positive or negative.
- 0 = negative
- 1 = positive

The model is trained only on `train.csv` and figured out on `public_test.csv`.

In [1]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import re
import random
import json

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [4]:
random.seed(7)
np.random.seed(7)
torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [5]:
train_df = pd.read_csv("train.csv")
public_test_df = pd.read_csv("public_test.csv")

print("Training shape:", train_df.shape)
print("Public test shape:", public_test_df.shape)

Training shape: (240, 5)
Public test shape: (400, 5)


In [6]:
train_df.head()

,id,text,label,label_name,source_file
0,pos_cv230_7428,"well , i'll admit when i first heard about thi...",1,positive,pos/cv230_7428.txt
1,pos_cv853_29233,my summer was recently saved by two very diffe...,1,positive,pos/cv853_29233.txt
2,pos_cv771_28665,in october of 1962 the united states found its...,1,positive,pos/cv771_28665.txt
3,pos_cv449_8785,this is a good year if you want plenty of sci-...,1,positive,pos/cv449_8785.txt
4,pos_cv130_17083,"while watching wes anderson's rushmore , it ma...",1,positive,pos/cv130_17083.txt


In [7]:
print("Training labels:")
print(train_df["label"].value_counts())

print("\nPublic test labels:")
print(public_test_df["label"].value_counts())


Training labels:
label
1    180
0     60
Name: count, dtype: int64

Public test labels:
label
1    200
0    200
Name: count, dtype: int64


In [8]:
train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=7,
    stratify=train_df["label"]
)

print("Training samples:", len(train_part))
print("Validation samples:", len(val_part))

print("\nTraining labels:")
print(train_part["label"].value_counts())

print("\nValidation labels:")
print(val_part["label"].value_counts())

Training samples: 192
Validation samples: 48

Training labels:
label
1    144
0     48
Name: count, dtype: int64

Validation labels:
label
1    36
0    12
Name: count, dtype: int64




This training dataset has around maybe 240 reviews of movies and with 60 negative reviews and 180 remaining positive reviews. The current ratio stands at about 1:3 right now. 

The public test set contains 400 reviews and is balanced with 200 positive and 200 negative reviews.
 
In order to utilize the training set I was able to creatw a stratified training model and a validation split. 
This makes sure that there is a balanced positive to negative ratio among botht the subsets. The public test kept sepereate and will not be used or implemented to train the model.

In [9]:
def tokenize(text):
    text = str(text).lower()
    return re.findall(r"[a-zA-Z']+", text)

In [10]:
example_review = train_part.iloc[0]["text"]

print("Original review:")
print(example_review[:300])

print("\nFirst 30 tokens:")
print(tokenize(example_review)[:30])

Original review:
in some regards , making a movie is like trying to stretch a rubber band as far as you can without breaking it . 
try too hard , and it snaps ; but too much reservation means someone else will come along and pull it farther . 
in simon birch , director mark steven johnson shows us just how to master

First 30 tokens:
['in', 'some', 'regards', 'making', 'a', 'movie', 'is', 'like', 'trying', 'to', 'stretch', 'a', 'rubber', 'band', 'as', 'far', 'as', 'you', 'can', 'without', 'breaking', 'it', 'try', 'too', 'hard', 'and', 'it', 'snaps', 'but', 'too']


In [11]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

word_to_id = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

word_counts = {}

for text in train_part["text"]:
    for word in tokenize(text):
        word_counts[word] = word_counts.get(word, 0) + 1

MIN_FREQUENCY = 2

for word, count in word_counts.items():
    if count >= MIN_FREQUENCY:
        word_to_id[word] = len(word_to_id)

id_to_word = {
    i: word for word, i in word_to_id.items()
}

print("Vocabulary size:", len(word_to_id))
print("PAD token ID:", word_to_id[PAD_TOKEN])
print("UNK token ID:", word_to_id[UNK_TOKEN])

Vocabulary size: 7305
PAD token ID: 0
UNK token ID: 1


In [12]:
MAX_LENGTH = 300

def encode_review(text, word_to_id, max_length=MAX_LENGTH):
    tokens = tokenize(text)

    token_ids = [
        word_to_id.get(word, word_to_id[UNK_TOKEN])
        for word in tokens
    ]

    token_ids = token_ids[:max_length]

    if len(token_ids) < max_length:
        token_ids += [word_to_id[PAD_TOKEN]] * (max_length - len(token_ids))

    return token_ids

In [13]:
encoded_review = encode_review(
    train_part.iloc[0]["text"],
    word_to_id
)

print("Encoded review length:", len(encoded_review))
print("First 30 IDs:")
print(encoded_review[:30])

Encoded review length: 300
First 30 IDs:
[2, 3, 1, 4, 5, 6, 7, 8, 9, 10, 11, 5, 12, 13, 14, 15, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 20, 25, 26, 22]


In [14]:
class ReviewDataset(Dataset):
    def __init__(self, dataframe, word_to_id):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].astype(int).tolist()
        self.word_to_id = word_to_id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoded = encode_review(
            self.texts[index],
            self.word_to_id
        )

        x = torch.tensor(encoded, dtype=torch.long)
        y = torch.tensor(self.labels[index], dtype=torch.long)

        return x, y

In [15]:
train_dataset = ReviewDataset(
    train_part,
    word_to_id
)

val_dataset = ReviewDataset(
    val_part,
    word_to_id
)

public_test_dataset = ReviewDataset(
    public_test_df,
    word_to_id
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Public test dataset:", len(public_test_dataset))

Training dataset: 192
Validation dataset: 48
Public test dataset: 400


In [16]:
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

public_test_loader = DataLoader(
    public_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [17]:
x_batch, y_batch = next(iter(train_loader))

print("Review batch shape:", x_batch.shape)
print("Label batch shape:", y_batch.shape)
print("Labels:", y_batch)

Review batch shape: torch.Size([8, 300])
Label batch shape: torch.Size([8])
Labels: tensor([1, 1, 1, 0, 1, 0, 1, 1])


In [18]:
class SentimentGRU(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128,
        dropout=0.5
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.output_layer = nn.Linear(
            hidden_dim,
            2
        )

    def forward(self, token_ids):
        embeddings = self.embedding(token_ids)

        _, hidden = self.gru(embeddings)

        final_hidden = hidden[-1]

        final_hidden = self.dropout(final_hidden)

        logits = self.output_layer(final_hidden)

        return logits

In [19]:
model = SentimentGRU(
    vocab_size=len(word_to_id),
    embedding_dim=100,
    hidden_dim=128,
    dropout=0.5
)

model = model.to(device)

print(model)

SentimentGRU(
  (embedding): Embedding(7305, 100, padding_idx=0)
  (gru): GRU(100, 128, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (output_layer): Linear(in_features=128, out_features=2, bias=True)
)


In [20]:
test_logits = model(x_batch.to(device))

print("Model output shape:", test_logits.shape)
print(test_logits)

Model output shape: torch.Size([8, 2])
tensor([[-0.2830,  0.0574],
        [ 0.1897,  0.1335],
        [-0.1359, -0.0578],
        [ 0.1141,  0.0380],
        [-0.0949, -0.0298],
        [-0.0507, -0.2097],
        [-0.3671, -0.0678],
        [ 0.1239,  0.1476]], grad_fn=<AddmmBackward0>)


In [21]:
train_labels = train_part["label"].values

class_counts = np.bincount(train_labels)

print("Class counts:", class_counts)

total_samples = len(train_labels)

class_weights = total_samples / (2 * class_counts)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class weights:", class_weights)

Class counts: [ 48 144]
Class weights: tensor([2.0000, 0.6667])


In [22]:
loss_function = nn.CrossEntropyLoss(
    weight=class_weights
)

LEARNING_RATE = 0.001

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Loss function:", loss_function)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)

Loss function: CrossEntropyLoss()
Optimizer: Adam
Learning rate: 0.001


In [23]:
def evaluate_model(model, data_loader):
    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)

            predictions = torch.argmax(logits, dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions)

    return accuracy, all_labels, all_predictions

In [24]:
val_accuracy, val_labels, val_predictions = evaluate_model(
    model,
    val_loader
)

print("Validation accuracy before training:", val_accuracy)

Validation accuracy before training: 0.3958333333333333


In [25]:
CHECKPOINT_DIR = Path("model_checkpoint")
CHECKPOINT_DIR.mkdir(exist_ok=True)

NUM_EPOCHS = 10
best_val_accuracy = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()

    total_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(x_batch)
        loss = loss_function(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    val_accuracy, _, _ = evaluate_model(model, val_loader)

    print(
        f"Epoch {epoch} | "
        f"Loss: {average_loss:.4f} | "
        f"Validation Accuracy: {val_accuracy:.4f}"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            CHECKPOINT_DIR / "sentiment_gru.pt"
        )

        print("Saved new best model.")

Epoch 1 | Loss: 0.7120 | Validation Accuracy: 0.5208
Saved new best model.
Epoch 2 | Loss: 0.6031 | Validation Accuracy: 0.6667
Saved new best model.
Epoch 3 | Loss: 0.5519 | Validation Accuracy: 0.6250
Epoch 4 | Loss: 0.4444 | Validation Accuracy: 0.6667
Epoch 5 | Loss: 0.2983 | Validation Accuracy: 0.6250
Epoch 6 | Loss: 0.1987 | Validation Accuracy: 0.4792
Epoch 7 | Loss: 0.1645 | Validation Accuracy: 0.6458
Epoch 8 | Loss: 0.1060 | Validation Accuracy: 0.5833
Epoch 9 | Loss: 0.0836 | Validation Accuracy: 0.6250
Epoch 10 | Loss: 0.0561 | Validation Accuracy: 0.5208





The training loss decreased in the epochs while the validation accuracy stopped improving at a steady rate which suggests that the model began to overfit the small training datasets. In order to reduce the effect of overfitting the model with the highest validation accuracy will be saved.

In [26]:
best_model = SentimentGRU(
    vocab_size=len(word_to_id),
    embedding_dim=100,
    hidden_dim=128,
    dropout=0.5
)

best_model.load_state_dict(
    torch.load(
        CHECKPOINT_DIR / "sentiment_gru.pt",
        map_location=device
    )
)

best_model = best_model.to(device)
best_model.eval()

best_val_accuracy, _, _ = evaluate_model(
    best_model,
    val_loader
)

print("Best validation accuracy:", best_val_accuracy)

Best validation accuracy: 0.6666666666666666


In [27]:
_, val_labels, val_predictions = evaluate_model(
    best_model,
    val_loader
)

print("Validation confusion matrix:")
print(confusion_matrix(val_labels, val_predictions))

Validation confusion matrix:
[[ 1 11]
 [ 5 31]]


In [28]:
public_accuracy, public_labels, public_predictions = evaluate_model(
    best_model,
    public_test_loader
)

print("Public test accuracy:", public_accuracy)

print("\nPublic test confusion matrix:")
print(confusion_matrix(public_labels, public_predictions))

Public test accuracy: 0.5575

Public test confusion matrix:
[[ 52 148]
 [ 29 171]]


In [29]:
predictions_df = pd.DataFrame({
    "id": public_test_df["id"],
    "predicted_label": public_predictions
})

predictions_df.to_csv(
    "public_test_predictions.csv",
    index=False
)

print(predictions_df.head())
print("\nNumber of predictions:", len(predictions_df))

                id  predicted_label
0  pos_cv696_29740                1
1  pos_cv669_22995                1
2   neg_cv963_7208                1
3   pos_cv182_7281                0
4  pos_cv162_10424                1

Number of predictions: 400


In [30]:
check_predictions = pd.read_csv("public_test_predictions.csv")

print(check_predictions.head())
print("\nColumns:", check_predictions.columns.tolist())
print("Rows:", len(check_predictions))
print("Prediction values:", sorted(check_predictions["predicted_label"].unique()))

                id  predicted_label
0  pos_cv696_29740                1
1  pos_cv669_22995                1
2   neg_cv963_7208                1
3   pos_cv182_7281                0
4  pos_cv162_10424                1

Columns: ['id', 'predicted_label']
Rows: 400
Prediction values: [np.int64(0), np.int64(1)]


In [31]:
checkpoint_data = {
    "model_state_dict": best_model.state_dict(),
    "word_to_id": word_to_id,
    "max_length": MAX_LENGTH,
    "embedding_dim": 100,
    "hidden_dim": 128,
    "dropout": 0.5
}

torch.save(
    checkpoint_data,
    CHECKPOINT_DIR / "model_checkpoint.pt"
)

print("Checkpoint saved.")

Checkpoint saved.


Training Techniques

This model has been trained using the adam optimizer and has a learning rate of 0.001 and a batch size consisting of 8. Weighted cross entrpy loss was utilized to help address the 3:1 training data imblances as the negative class has a larger loss weight. This model was trained for 10 epochs and the validation accuracy was checked after each epoch implemented asnd the model that has the highest accuracy was saved. 


Evaluation Results

The best validation accuracy was around 66.67%. The final saved model has a public test accuracy of 55.75%.

The public test confusion matrix was as shown below and displays that the results show us that the model predicted positive reviews more frequently than the negative reviews.


| | Predicted Negative | Predicted Positive |
|---|---:|---:|
| Actual Negative | 52 | 148 |
| Actual Positive | 29 | 171 |



Use of AI


I utilized generative AI as a learning assistant during this assignment as it helped explain to me PyTorch concepts, organize my notebook and provided guidence and feeback for debugging my training models. I used prompts that would aid my process of learning and teach me methods rather than hand me the answer. I inputted promts such as - "Explain how a GRU can be used for sentiment classification." and "How do I save and reload my trained model without retraining?".
The model was trained and evaluated using the provided course datasets, and the results reported in this notebook were generated by running the code in this notebook.
